# Suggestion algorithm bakeoff

Supervisor report: fair comparison of ranking algorithms for **top-4 suggestion ranking** on Dataset C.

## Fixed contract

| Item | Value |
|------|--------|
| Dataset | `data/processed/suggestion_dataset.parquet` |
| Splits | frozen `train` / `val` / `test` |
| Seed | `42` |
| Features | 96-d via `encode_dataset_row` |
| Neural backbone | same MLP (`suggestion_model.SuggestionRanker`) |
| Accuracy | **NDCG@4**, **MAP@4** |
| Efficiency | `train_seconds`, `infer_ms_per_sample`, `n_params` |
| Metrics | `data/processed/bakeoff/metrics.json` |

## Algorithms

| Tag | Category | Script |
|-----|----------|--------|
| `bce` | pointwise_supervised | `train_bce.py` |
| `reinforce_ndcg` | listwise_rl | `train_reinforce_ndcg.py` |
| `bpr_pairwise` | pairwise_ltr | `train_bpr_pairwise.py` |
| `listnet` | listwise_supervised | `train_listnet.py` |
| `approx_ndcg` | metric_listwise | `train_approx_ndcg.py` |
| `logistic` | classical_linear | `train_logistic.py` |
| `lightgbm` | classical_gbdt | `train_lightgbm.py` |

Figures are saved to `data/processed/bakeoff/figures/`.


In [1]:
from pathlib import Path
import json

import pandas as pd
import matplotlib

# Headless-safe when executing via nbconvert / CI
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

# Resolve repo root whether cwd is notebooks/ or repo root
_cwd = Path.cwd().resolve()
if (_cwd / "data" / "processed" / "bakeoff" / "metrics.json").is_file():
    REPO = _cwd
elif (_cwd.parent / "data" / "processed" / "bakeoff" / "metrics.json").is_file():
    REPO = _cwd.parent
else:
    REPO = Path("..").resolve()

METRICS_PATH = REPO / "data" / "processed" / "bakeoff" / "metrics.json"
FIG_DIR = REPO / "data" / "processed" / "bakeoff" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "axes.grid": True,
    "grid.alpha": 0.25,
    "font.size": 11,
})

payload = json.loads(METRICS_PATH.read_text(encoding="utf-8"))
models = payload.get("models") or {}
df = pd.DataFrame(list(models.values())) if models else pd.DataFrame()
if df.empty:
    raise RuntimeError(f"No models in {METRICS_PATH} — run Phase 2 trainers first.")

df = df.sort_values("test_ndcg@4", ascending=False).reset_index(drop=True)
display_cols = [
    "tag", "category",
    "val_ndcg@4", "test_ndcg@4", "val_map@4", "test_map@4",
    "train_seconds", "infer_ms_per_sample", "n_params", "seed",
]
display_cols = [c for c in display_cols if c in df.columns]
print(f"Loaded {len(df)} models from {METRICS_PATH.relative_to(REPO)} (sorted by test NDCG@4)")
df[display_cols]


Loaded 7 models from data/processed/bakeoff/metrics.json (sorted by test NDCG@4)


,tag,category,val_ndcg@4,test_ndcg@4,val_map@4,test_map@4,train_seconds,infer_ms_per_sample,n_params,seed
0,lightgbm,classical_gbdt,0.900502,0.916648,0.991319,0.993421,0.9964,3.349097,54232,42
1,listnet,listwise_supervised,0.908223,0.910134,0.980903,0.971491,0.7998,0.001158,35120,42
2,logistic,classical_linear,0.890386,0.877286,0.982639,0.975877,0.1600,0.045438,4608,42
3,reinforce_ndcg,listwise_rl,0.831261,0.833134,0.933449,0.936404,1.3610,0.001253,35120,42
4,approx_ndcg,metric_listwise,0.821668,0.805035,0.917824,0.883041,0.7683,0.001213,35120,42
5,bce,pointwise_supervised,0.757908,0.785082,0.913194,0.925439,0.4097,0.001389,35120,42
6,bpr_pairwise,pairwise_ltr,0.728090,0.777715,0.896991,0.909357,1.3027,0.001214,35120,42


## Accuracy (test NDCG@4 / MAP@4)


In [1]:
plot_df = df.set_index("tag")
fig, ax = plt.subplots(figsize=(10, 4.5))
plot_df[["test_ndcg@4", "test_map@4"]].plot(
    kind="bar", ax=ax, color=["#2c5f7c", "#c47b2d"], width=0.75,
)
ax.set_title("Accuracy comparison (test)")
ax.set_ylabel("Score")
ax.set_xlabel("")
ax.set_ylim(0, 1.05)
ax.legend(["test NDCG@4", "test MAP@4"], loc="lower right")
ax.tick_params(axis="x", rotation=35)
fig.tight_layout()
out = FIG_DIR / "accuracy_ndcg_map.png"
fig.savefig(out, dpi=200, bbox_inches="tight")
print(f"Wrote {out.relative_to(REPO)}")
plt.show()


NameError: name 'df' is not defined

## Efficiency (train time / inference latency)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
plot_df["train_seconds"].plot(kind="bar", ax=axes[0], color="#2c5f7c", width=0.7)
axes[0].set_title("Train time")
axes[0].set_ylabel("seconds")
axes[0].set_xlabel("")
axes[0].tick_params(axis="x", rotation=35)

plot_df["infer_ms_per_sample"].plot(kind="bar", ax=axes[1], color="#c47b2d", width=0.7)
axes[1].set_title("Inference latency")
axes[1].set_ylabel("ms / sample")
axes[1].set_xlabel("")
axes[1].tick_params(axis="x", rotation=35)

fig.tight_layout()
out = FIG_DIR / "efficiency_train_infer.png"
fig.savefig(out, dpi=200, bbox_inches="tight")
print(f"Wrote {out.relative_to(REPO)}")
plt.show()


Wrote data/processed/bakeoff/figures/efficiency_train_infer.png


/var/folders/c8/58b5yypd5cnb9pk9yrk5qqpw0000gn/T/ipykernel_2913/1874411159.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Accuracy vs efficiency

Best region: **high test NDCG@4**, **low train time** (upper-left).


In [4]:
fig, ax = plt.subplots(figsize=(8, 5.5))
ax.scatter(
    df["train_seconds"], df["test_ndcg@4"],
    s=90, c="#2c5f7c", zorder=3, edgecolors="white", linewidths=0.8,
)
for _, r in df.iterrows():
    ax.annotate(
        r["tag"],
        (r["train_seconds"], r["test_ndcg@4"]),
        textcoords="offset points",
        xytext=(6, 5),
        fontsize=9,
    )
ax.set_xlabel("Train time (s)")
ax.set_ylabel("test NDCG@4")
ax.set_title("Accuracy vs efficiency (best = high NDCG, low time)")
fig.tight_layout()
out = FIG_DIR / "accuracy_vs_efficiency.png"
fig.savefig(out, dpi=200, bbox_inches="tight")
print(f"Wrote {out.relative_to(REPO)}")
plt.show()


Wrote data/processed/bakeoff/figures/accuracy_vs_efficiency.png


/var/folders/c8/58b5yypd5cnb9pk9yrk5qqpw0000gn/T/ipykernel_2913/4054475620.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Generalization (val vs test NDCG@4)


In [3]:
fig, ax = plt.subplots(figsize=(10, 4.5))
plot_df[["val_ndcg@4", "test_ndcg@4"]].plot(
    kind="bar", ax=ax, color=["#5a7a8c", "#2c5f7c"], width=0.75,
)
ax.set_title("Generalization: val vs test NDCG@4")
ax.set_ylabel("NDCG@4")
ax.set_xlabel("")
ax.set_ylim(0, 1.05)
ax.legend(["val NDCG@4", "test NDCG@4"], loc="lower right")
ax.tick_params(axis="x", rotation=35)
fig.tight_layout()
out = FIG_DIR / "generalization_val_test.png"
fig.savefig(out, dpi=200, bbox_inches="tight")
print(f"Wrote {out.relative_to(REPO)}")
plt.show()


Wrote data/processed/bakeoff/figures/generalization_val_test.png


/var/folders/c8/58b5yypd5cnb9pk9yrk5qqpw0000gn/T/ipykernel_34704/1415383478.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Summary table


In [6]:
summary_cols = [
    "tag", "category",
    "test_ndcg@4", "test_map@4", "val_ndcg@4", "val_map@4",
    "train_seconds", "infer_ms_per_sample", "n_params",
]
summary = df[summary_cols].copy()
for c in ["test_ndcg@4", "test_map@4", "val_ndcg@4", "val_map@4"]:
    summary[c] = summary[c].map(lambda x: f"{x:.4f}")
summary["train_seconds"] = summary["train_seconds"].map(lambda x: f"{x:.3f}")
summary["infer_ms_per_sample"] = summary["infer_ms_per_sample"].map(lambda x: f"{x:.4f}")

csv_path = FIG_DIR / "summary.csv"
df[summary_cols].to_csv(csv_path, index=False)
print(f"Wrote {csv_path.relative_to(REPO)}")

fig, ax = plt.subplots(figsize=(14, 0.55 + 0.4 * len(summary)))
ax.axis("off")
table = ax.table(
    cellText=summary.values,
    colLabels=list(summary.columns),
    loc="center",
    cellLoc="center",
)
table.auto_set_font_size(False)
table.set_fontsize(8)
table.scale(1.0, 1.35)
for (row, col), cell in table.get_celld().items():
    if row == 0:
        cell.set_facecolor("#2c5f7c")
        cell.set_text_props(color="white", weight="bold")
    elif row % 2 == 0:
        cell.set_facecolor("#f2f5f7")
ax.set_title("Bakeoff summary (sorted by test NDCG@4)", pad=12)
fig.tight_layout()
out = FIG_DIR / "summary_table.png"
fig.savefig(out, dpi=200, bbox_inches="tight")
print(f"Wrote {out.relative_to(REPO)}")
plt.show()


Wrote data/processed/bakeoff/figures/summary.csv


Wrote data/processed/bakeoff/figures/summary_table.png


/var/folders/c8/58b5yypd5cnb9pk9yrk5qqpw0000gn/T/ipykernel_2913/3836749034.py:38: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Decision rule (prove suitability)

1. **Primary accuracy:** highest **test NDCG@4**.
2. **MAP guard:** do not accept a large drop in test MAP@4 vs BCE.
3. **Efficiency tie-break:** if two models are within **~0.02 NDCG**, prefer faster train + faster inference.
4. Existing ship rule (RL vs BCE): promote RL only if `val_ndcg >= bce.val_ndcg` and `test_ndcg >= bce.test_ndcg - 0.02`.


In [6]:
winner = df.iloc[0]
bce = df.loc[df["tag"] == "bce"].iloc[0] if (df["tag"] == "bce").any() else None
rl = df.loc[df["tag"] == "reinforce_ndcg"].iloc[0] if (df["tag"] == "reinforce_ndcg").any() else None

print("=== Bakeoff decision ===")
print(
    f"Winner by test NDCG@4: {winner['tag']} "
    f"({winner['category']})  test_ndcg@4={winner['test_ndcg@4']:.4f}  "
    f"test_map@4={winner['test_map@4']:.4f}  train_s={winner['train_seconds']:.3f}"
)

# Within 0.02 of winner → prefer faster
near = df[df["test_ndcg@4"] >= float(winner["test_ndcg@4"]) - 0.02].copy()
near = near.sort_values(["train_seconds", "infer_ms_per_sample"])
efficient = near.iloc[0]
if efficient["tag"] != winner["tag"]:
    print(
        f"Efficiency tie-break (within 0.02 NDCG): prefer {efficient['tag']} "
        f"(train_s={efficient['train_seconds']:.3f}, "
        f"infer_ms={efficient['infer_ms_per_sample']:.4f})"
    )
else:
    print("No efficiency override — winner also among the fastest near the top.")

if bce is not None:
    map_delta = float(winner["test_map@4"]) - float(bce["test_map@4"])
    print(f"MAP guard vs BCE: Δtest_map@4 = {map_delta:+.4f}")

if bce is not None and rl is not None:
    ship = (
        float(rl["val_ndcg@4"]) >= float(bce["val_ndcg@4"])
        and float(rl["test_ndcg@4"]) >= float(bce["test_ndcg@4"]) - 0.02
    )
    print(
        f"RL vs BCE ship gate: {'PASS' if ship else 'FAIL'} "
        f"(rl val/test={rl['val_ndcg@4']:.4f}/{rl['test_ndcg@4']:.4f}, "
        f"bce val/test={bce['val_ndcg@4']:.4f}/{bce['test_ndcg@4']:.4f})"
    )

print(f"\nFigures directory: {FIG_DIR.relative_to(REPO)}")


=== Bakeoff decision ===
Winner by test NDCG@4: lightgbm (classical_gbdt)  test_ndcg@4=0.9166  test_map@4=0.9934  train_s=0.996
Efficiency tie-break (within 0.02 NDCG): prefer listnet (train_s=0.800, infer_ms=0.0012)
MAP guard vs BCE: Δtest_map@4 = +0.0680
RL vs BCE ship gate: PASS (rl val/test=0.8313/0.8331, bce val/test=0.7579/0.7851)

Figures directory: data/processed/bakeoff/figures
